# SentinelAPI: confidence-gated tool selection

This notebook demonstrates a reusable **Strands Decider 2B tool-selection pattern**. A natural-language request is mapped to one validated choice: `idor_probe`, `exposure_probe`, or `no_probe`. Confidence controls whether the decision abstains, needs review, or is eligible to run after explicit approval.

**Safety boundary:** probes are fixed Python functions that call only the bundled synthetic API at `127.0.0.1:8088`. The notebook accepts no target URL, does not test external systems, and does not print synthetic secret-field values. Keep `.env` and all API keys out of Git.

The optional explanation cell calls a Strands agent backed by Amazon Bedrock. It needs AWS credentials and model invocation permission; the deterministic finding still works without that optional explanation.

## Before you run

1. Open the SentinelAPI project folder in VS Code and select its `.venv` Python kernel for this notebook.
2. Install the project requirements if needed: ` .\.venv\Scripts\python.exe -m pip install -r requirements.txt `.
3. Configure the hosted Decider endpoint and replacement API key in your local `.env` file. Never put the key in a notebook cell.
4. In a separate VS Code terminal, start the local synthetic API: ` .\.venv\Scripts\python.exe -m uvicorn sandbox_api:app --host 127.0.0.1 --port 8088 `.

Run the cells from top to bottom. The probe cell starts with execution disabled; review the decision and explicitly change the approval flag before running it.

In [ ]:
from pathlib import Path
import os
import sys

# Find the project root whether Jupyter starts in the repo or notebooks/ folder.
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'decision_pattern.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the SentinelAPI project so decision_pattern.py can be found.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print('Credentials are loaded by the project from local environment settings; this cell does not display them.')

In [ ]:
import requests

SANDBOX_HEALTH_URL = 'http://127.0.0.1:8088/health'
response = requests.get(SANDBOX_HEALTH_URL, timeout=3)
response.raise_for_status()
health = response.json()
assert health.get('environment') == 'synthetic-local-sandbox', 'Unexpected service response; stopping for safety.'
print('Local synthetic sandbox is ready:', health)

## Ask Decider to select a bounded action

Try one of the included examples. Do not enter a URL or a real service name: SentinelAPI rejects URL targets and only knows its fixed local sandbox. This cell makes a Decider request but does not run a probe.

In [ ]:
from decision_pattern import choose_probe

user_request = 'Check whether the public profile response exposes internal fields.'
decision = choose_probe(user_request)

print('Request:', user_request)
print('Choice:', decision.choice)
print(f'Confidence: {decision.confidence:.1%}')
print(f'Measured Decider latency: {decision.latency_ms:.0f} ms')
print('Model:', decision.model)
print('Choice distribution:', decision.probabilities)

## Apply the confidence and human-review gate

Policy: `no_probe` or confidence below 50% abstains; 50–69% requires explicit review; 70% or higher is eligible after a deliberate run action. Confidence is not proof of correctness. The next cell will not execute anything until `RUN_SELECTED_PROBE` is changed to `True`. For a 50–69% decision, also set `APPROVE_MEDIUM_CONFIDENCE` to `True` after reviewing the choice.

In [ ]:
from scanner import execute_probe

AUTO_RUN_CONFIDENCE = 0.70
REVIEW_CONFIDENCE = 0.50
APPROVE_MEDIUM_CONFIDENCE = False  # Set True only after reviewing a 50-69% choice.
RUN_SELECTED_PROBE = False         # Deliberate opt-in; set True only to run the fixed localhost probe.
finding = None

if decision.choice == 'no_probe':
    print('Abstained: Decider selected no_probe. No sandbox request was made.')
elif decision.confidence < REVIEW_CONFIDENCE:
    print('Abstained: confidence is below 50%. No sandbox request was made.')
elif decision.confidence < AUTO_RUN_CONFIDENCE and not APPROVE_MEDIUM_CONFIDENCE:
    print('Human review required for this 50-69% choice. Review it, then set APPROVE_MEDIUM_CONFIDENCE = True if you approve.')
elif not RUN_SELECTED_PROBE:
    print('Choice is eligible, but no probe ran. Review the choice, set RUN_SELECTED_PROBE = True, then rerun this cell to explicitly approve the local probe.')
else:
    finding = execute_probe(decision.choice)
    print('Probe ran against the fixed synthetic localhost sandbox.')
    print(finding.to_dict())

## Optional: explain the observed finding with Strands on Bedrock

This last step is optional. It only receives the deterministic finding and cannot select or run another probe. If your AWS account cannot invoke the selected Bedrock model, the local finding remains available; the error below intentionally avoids printing credentials.

In [ ]:
if finding is None:
    print('No finding is available to explain. Approve and run a sandbox probe first.')
else:
    from agent import explain_finding
    decision_summary = {'choice': decision.choice, 'confidence': decision.confidence}
    try:
        print(explain_finding(finding.to_dict(), decision_summary))
    except Exception as exc:
        print(f'Bedrock explanation unavailable ({type(exc).__name__}). The deterministic sandbox finding above is still valid.')

## Reusable pattern summary

1. **Bound the decision:** Decider selects from a typed, allowlisted set.
2. **Validate before acting:** reject malformed or unknown choices.
3. **Scale oversight with confidence:** abstain, request review, or allow a deliberate run.
4. **Keep tools fixed:** the request cannot choose a host, path, or arbitrary arguments.
5. **Separate explanation from action:** Bedrock explains observed evidence but cannot change the action.

Pattern tag: `#tool-selection`. All findings in this notebook concern synthetic demo data on localhost.